# Raw Canny: development experiment and evaluation

This notebook runs only the 139-image development split. Canny is applied independently to the aligned reference and defective grayscale images, followed by absolute edge-map difference and direct external-contour extraction. There is no blur, resizing, morphology, dilation, filtering, or merging.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'data' / 'dataset_split.csv').is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from algorithms.canny import detect_canny
from algorithms.common import load_image
from algorithms.evaluation import parse_voc_boxes
from scripts.development.run_canny_development import (
    VALIDATION_CANDIDATE_PAIRS, evaluate_records, load_development_rows,
    summarise, write_results,
)
plt.rcParams['figure.dpi'] = 110

## 1. Development baseline—not a frozen winner

The `(50, 150)` pair is used to implement and diagnose raw Canny. The recorded candidate pairs will later be compared on validation. IoU 0.50 is a provisional common evaluation rule, not a Canny parameter.

In [ ]:
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'dataset_split.csv'
RESULTS_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'canny_development.csv'
SUMMARY_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'canny_development_summary.json'
BOXES_DIR = PROJECT_ROOT / 'outputs' / 'metrics' / 'canny_development_boxes'
LOW_THRESHOLD, HIGH_THRESHOLD = 50, 150
APERTURE_SIZE = 3
L2_GRADIENT = False
IOU_THRESHOLD = 0.50
RUN_DEVELOPMENT = False

development_baseline = {
    'low_threshold': LOW_THRESHOLD,
    'high_threshold': HIGH_THRESHOLD,
    'aperture_size': APERTURE_SIZE,
    'l2_gradient': L2_GRADIENT,
    'pipeline': 'Canny(reference), Canny(defective), absolute edge difference',
    'post_processing': 'none',
    'evaluation_iou': IOU_THRESHOLD,
}
display(pd.Series(development_baseline, name='setting').to_frame())
display(pd.DataFrame(
    VALIDATION_CANDIDATE_PAIRS, columns=['low_threshold', 'high_threshold']
).rename_axis('validation_candidate'))

## 2. Manifest and development-split verification

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)
development_df = manifest_df.loc[manifest_df['split'].eq('development')].copy()
assert len(manifest_df) == 693
assert len(development_df) == 139
assert set(development_df['split']) == {'development'}
assert development_df['image_id'].is_unique

split_counts = manifest_df['split'].value_counts().reindex(
    ['development', 'validation', 'test']
)
class_counts = development_df['defect_class'].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
split_counts.plot.bar(ax=axes[0], color=['#4C78A8', '#F58518', '#54A24B'])
axes[0].set_title('Authoritative manifest split')
axes[0].set_ylabel('Images')
axes[0].tick_params(axis='x', rotation=0)
class_counts.plot.bar(ax=axes[1], color='#F58518')
axes[1].set_title('Development class distribution')
axes[1].set_ylabel('Images')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 3. Execute or reload development results

Set `RUN_DEVELOPMENT = True` to recompute all full-resolution development records. Complete unfiltered boxes are stored as typed NumPy sidecars.

In [ ]:
outputs_exist = RESULTS_PATH.is_file() and SUMMARY_PATH.is_file()
if RUN_DEVELOPMENT or not outputs_exist:
    rows = load_development_rows(MANIFEST_PATH)
    results = evaluate_records(
        rows, IOU_THRESHOLD, workers=1, boxes_directory=BOXES_DIR,
        low_threshold=LOW_THRESHOLD, high_threshold=HIGH_THRESHOLD,
        aperture_size=APERTURE_SIZE, l2_gradient=L2_GRADIENT,
    )
    summary = summarise(
        results, IOU_THRESHOLD, LOW_THRESHOLD, HIGH_THRESHOLD,
        APERTURE_SIZE, L2_GRADIENT,
    )
    write_results(results, RESULTS_PATH)
    SUMMARY_PATH.write_text(
        json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8'
    )
else:
    print('Loading existing development results.')

results_df = pd.read_csv(RESULTS_PATH)
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
assert len(results_df) == 139
assert set(results_df['split']) == {'development'}
assert set(results_df['status']) == {'success'}
assert summary['errors'] == 0
results_df.head()

## 4. Overall and class-level evaluation

In [ ]:
overall_df = pd.Series(summary['overall_box_metrics'], name='raw_canny').to_frame()
runtime_df = pd.Series(summary['runtime_ms'], name='raw_canny').to_frame()
class_metrics_df = pd.DataFrame(summary['box_metrics_by_class']).T
display(overall_df)
display(runtime_df)
display(class_metrics_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
class_metrics_df['recall'].plot.bar(ax=axes[0], color='#F58518')
axes[0].set_title('Recall by class at IoU 0.50')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=30)
class_metrics_df[['precision', 'f1_score']].plot.bar(ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Precision and F1 (log scale)')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 5. Fragmentation and runtime diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
results_df.boxplot(column='edge_difference_percentage', by='defect_class', ax=axes[0], rot=30)
axes[0].set_title('Differing edge pixels (%)')
results_df.boxplot(column='predicted_count', by='defect_class', ax=axes[1], rot=30)
axes[1].set_yscale('symlog', linthresh=1)
axes[1].set_title('Direct contours (log scale)')
results_df.boxplot(column='processing_time_ms', by='defect_class', ax=axes[2], rot=30)
axes[2].set_title('Runtime (ms)')
fig.suptitle('')
plt.tight_layout()
plt.show()
results_df.groupby('defect_class').agg(
    images=('image_id', 'size'),
    median_regions=('predicted_count', 'median'),
    maximum_regions=('predicted_count', 'max'),
    mean_runtime_ms=('processing_time_ms', 'mean'),
)

## 6. Representative raw edge-map visualisations

In [ ]:
def show_canny_result(image_id, maximum_drawn_boxes=500):
    manifest_row = development_df.loc[development_df['image_id'].eq(image_id)].iloc[0]
    result_row = results_df.loc[results_df['image_id'].eq(image_id)].iloc[0]
    reference = load_image(PROJECT_ROOT / manifest_row['reference_path'])
    defective = load_image(PROJECT_ROOT / manifest_row['image_path'])
    ground_truth = parse_voc_boxes(PROJECT_ROOT / manifest_row['annotation_path'])
    detection = detect_canny(
        reference, defective, LOW_THRESHOLD, HIGH_THRESHOLD, APERTURE_SIZE, L2_GRADIENT
    )
    overlay = defective.copy()
    for box in ground_truth:
        cv2.rectangle(overlay, (int(box['xmin']), int(box['ymin'])),
                      (int(box['xmax']), int(box['ymax'])), (0, 0, 255), 4)
    boxes_drawn = len(detection.boxes) <= maximum_drawn_boxes
    if boxes_drawn:
        for box in detection.boxes:
            cv2.rectangle(overlay, (int(box['xmin']), int(box['ymin'])),
                          (int(box['xmax']), int(box['ymax'])), (0, 255, 0), 2)
    panels = [
        (reference, 'Reference', True),
        (detection.reference_edges, 'Reference edges', False),
        (detection.defective_edges, 'Defective edges', False),
        (detection.edge_difference, 'Raw edge difference', False),
        (overlay, 'GT red; predictions green' if boxes_drawn else 'GT red; boxes omitted', True),
    ]
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    for axis, (image, title, color) in zip(axes, panels):
        axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB) if color else image,
                    cmap=None if color else 'gray')
        axis.set_title(title)
        axis.axis('off')
    fig.suptitle(f"{image_id}: {int(result_row['predicted_count']):,} contours")
    plt.tight_layout()
    plt.show()

best_image_id = results_df.sort_values(
    ['mean_matched_iou', 'recall'], ascending=False
).iloc[0]['image_id']
fragmented_image_id = results_df.sort_values('predicted_count', ascending=False).iloc[0]['image_id']
show_canny_result(best_image_id)
show_canny_result(fragmented_image_id)

## 7. Development conclusion

This baseline is development-complete after the saved results are regenerated. It does not select the winning thresholds. Compare the recorded candidates on validation, freeze one pair, and keep the test split untouched until all four algorithms are frozen.